<a href="https://colab.research.google.com/github/angelrecalde2024/Power-System-Planning-and-Transmission-Design-2026/blob/main/INGP1118_PowerFlow_AC_decoupledDC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================
# 1) Install & imports
# =========================
!pip -q install pypower openpyxl pandas

import numpy as np
import pandas as pd

from pypower.api import runpf, rundcpf, ppoption
from pypower.idx_bus import BUS_I, BUS_TYPE, PD, QD, GS, BS, BUS_AREA, VM, VA, BASE_KV, ZONE, VMAX, VMIN
from pypower.idx_gen import GEN_BUS, PG, QG, QMAX, QMIN, VG, MBASE, GEN_STATUS, PMAX, PMIN
from pypower.idx_brch import F_BUS, T_BUS, BR_R, BR_X, BR_B, RATE_A, TAP, SHIFT, BR_STATUS, PF, PT

# =========================
# 2) User settings
# =========================
EXCEL_PATH = "/content/Data6busSystemTEP.xlsx"

SLACK_BUS = 1          # <-- user-selectable slack bus number
BASE_MVA  = 100.0      # typical planning base

BASE_KV_DEFAULT = 230.0  # not critical for PF math; used mainly for reporting

# =========================
# 3) Read Excel data
# =========================
gen_df  = pd.read_excel(EXCEL_PATH, sheet_name="generator")
load_df = pd.read_excel(EXCEL_PATH, sheet_name="load")
line_df = pd.read_excel(EXCEL_PATH, sheet_name="lines")

# Normalize/clean expected column names (robust to small variations)
def require_cols(df, required, sheet_name):
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in sheet '{sheet_name}': {missing}\nFound: {list(df.columns)}")

require_cols(gen_df,  ["generator bus", "Pmin", "Pmax", "Qmax", "Qmin"], "generator")
require_cols(load_df, ["load bus", "Pload", "Qload"], "load")
require_cols(line_df, ["from", "to", "R", "X", "Bcap", "Flow limit"], "lines")

# Optional voltage guess column name handling
# User stated: "initial voltage guess in p.u."
vguess_col_candidates = [c for c in gen_df.columns if "volt" in c.lower() or "v" == c.lower().strip()]
VGUESS_COL = vguess_col_candidates[0] if vguess_col_candidates else None

# Bus set
all_buses = sorted(set(load_df["load bus"].astype(int).tolist()) |
                   set(gen_df["generator bus"].astype(int).tolist()) |
                   set(line_df["from"].astype(int).tolist()) |
                   set(line_df["to"].astype(int).tolist()))
nb = len(all_buses)

if SLACK_BUS not in all_buses:
    raise ValueError(f"Selected SLACK_BUS={SLACK_BUS} not found in data buses: {all_buses}")

bus_index = {b: i for i, b in enumerate(all_buses)}  # mapping bus number -> row

# =========================
# 4) Build MATPOWER case matrices
# =========================
# ---- BUS matrix ----
# MATPOWER bus: [BUS_I BUS_TYPE PD QD GS BS BUS_AREA VM VA BASE_KV ZONE VMAX VMIN]
bus = np.zeros((nb, 13), dtype=float)
for i, b in enumerate(all_buses):
    bus[i, BUS_I]     = b
    bus[i, BUS_TYPE]  = 1  # PQ by default
    bus[i, PD]        = 0.0
    bus[i, QD]        = 0.0
    bus[i, GS]        = 0.0
    bus[i, BS]        = 0.0
    bus[i, BUS_AREA]  = 1
    bus[i, VM]        = 1.0
    bus[i, VA]        = 0.0
    bus[i, BASE_KV]   = BASE_KV_DEFAULT
    bus[i, ZONE]      = 1
    bus[i, VMAX]      = 1.10
    bus[i, VMIN]      = 0.90

# Add loads
for _, r in load_df.iterrows():
    b = int(r["load bus"])
    i = bus_index[b]
    bus[i, PD] += float(r["Pload"])
    bus[i, QD] += float(r["Qload"])

# ---- GEN matrix ----
# MATPOWER gen: [GEN_BUS PG QG QMAX QMIN VG MBASE GEN_STATUS PMAX PMIN]
ng = len(gen_df)
gen = np.zeros((ng, 10), dtype=float)

# Voltage setpoints from guess if available, else 1.0
def vset_from_row(row):
    if VGUESS_COL is None:
        return 1.0
    val = row[VGUESS_COL]
    try:
        return float(val) if not pd.isna(val) else 1.0
    except:
        return 1.0

for k, (_, r) in enumerate(gen_df.iterrows()):
    gb = int(r["generator bus"])
    gen[k, GEN_BUS]    = gb
    gen[k, PG]         = 0.0   # will set below
    gen[k, QG]         = 0.0
    gen[k, QMAX]       = float(r["Qmax"])
    gen[k, QMIN]       = float(r["Qmin"])
    gen[k, VG]         = vset_from_row(r)
    gen[k, MBASE]      = BASE_MVA
    gen[k, GEN_STATUS] = 1.0
    gen[k, PMAX]       = float(r["Pmax"])
    gen[k, PMIN]       = float(r["Pmin"])

# ---- Set bus types based on generators & slack selection ----
gen_buses = sorted(gen_df["generator bus"].astype(int).unique().tolist())
for b in gen_buses:
    i = bus_index[b]
    bus[i, BUS_TYPE] = 2  # PV
    # set initial VM for PV buses
    # Use first generator's VG at that bus (if multiple, keep first)
    vg = gen[gen[:, GEN_BUS] == b, VG]
    if len(vg) > 0:
        bus[i, VM] = float(vg[0])

# Slack bus
bus[bus_index[SLACK_BUS], BUS_TYPE] = 3
# Slack bus magnitude initial guess from any generator at slack (if exists)
if SLACK_BUS in gen_buses:
    vg = gen[gen[:, GEN_BUS] == SLACK_BUS, VG]
    if len(vg) > 0:
        bus[bus_index[SLACK_BUS], VM] = float(vg[0])

# ---- Simple initial PG dispatch so PF is well-posed ----
total_load = bus[:, PD].sum()
slack_mask = (gen[:, GEN_BUS] == SLACK_BUS)
non_slack_gens = np.where(~slack_mask)[0]
slack_gens     = np.where(slack_mask)[0]

if len(slack_gens) == 0:
    raise ValueError(f"No generator at SLACK_BUS={SLACK_BUS}. Add one generator at the slack bus in the Excel file.")

# Allocate non-slack PG evenly (clipped to [PMIN, PMAX]), slack balances the rest.
if len(non_slack_gens) > 0:
    share = total_load / (len(non_slack_gens) + 1)  # +1 leaves room for slack
    for idx in non_slack_gens:
        gen[idx, PG] = np.clip(share, gen[idx, PMIN], gen[idx, PMAX])

slack_needed = total_load - gen[non_slack_gens, PG].sum()
# Put all balance on the first slack generator
s0 = slack_gens[0]
gen[s0, PG] = np.clip(slack_needed, gen[s0, PMIN], gen[s0, PMAX])

# If slack clipped, PF may still solve but mismatch exists; better to warn.
if not (gen[s0, PMIN] - 1e-6 <= slack_needed <= gen[s0, PMAX] + 1e-6):
    print("WARNING: Slack required PG is outside slack limits; clipping applied.",
          f"Required={slack_needed:.3f} MW, Clipped={gen[s0, PG]:.3f} MW")

# ---- BRANCH matrix ----
# MATPOWER branch (at least first 13 cols):
# [F_BUS T_BUS BR_R BR_X BR_B RATE_A RATE_B RATE_C TAP SHIFT BR_STATUS ANGMIN ANGMAX]
nl = len(line_df)
branch = np.zeros((nl, 13), dtype=float)

for k, (_, r) in enumerate(line_df.iterrows()):
    fb = int(r["from"])
    tb = int(r["to"])
    branch[k, F_BUS]     = fb
    branch[k, T_BUS]     = tb
    branch[k, BR_R]      = float(r["R"])
    branch[k, BR_X]      = float(r["X"])
    branch[k, BR_B]      = float(r["Bcap"])  # total line charging susceptance (p.u.)
    branch[k, RATE_A]    = float(r["Flow limit"])  # MVA (≈ MW for DC & HV)
    branch[k, TAP]       = 0.0
    branch[k, SHIFT]     = 0.0
    branch[k, BR_STATUS] = 1.0
    branch[k, 11]        = -360.0  # ANGMIN
    branch[k, 12]        =  360.0  # ANGMAX

# ---- GENCOST (dummy, not used in PF) ----
# Polynomial cost model: [MODEL STARTUP SHUTDOWN NCOST c(n-1) ... c0]
gencost = np.zeros((ng, 7), dtype=float)
gencost[:, 0] = 2  # polynomial
gencost[:, 3] = 3  # quadratic (3 coeffs)
# If a,b,c exist, place them; else keep zeros
for k, (_, r) in enumerate(gen_df.iterrows()):
    if all(col in gen_df.columns for col in ["a", "b", "c"]):
        # MATPOWER uses c2, c1, c0 ordering for quadratic
        gencost[k, 4] = float(r["a"])
        gencost[k, 5] = float(r["b"])
        gencost[k, 6] = float(r["c"])

ppc = {
    "version": "2",
    "baseMVA": BASE_MVA,
    "bus": bus,
    "gen": gen,
    "branch": branch,
    "gencost": gencost
}

# =========================
# 5) Run AC PF and DC PF
# =========================
ppopt = ppoption(VERBOSE=0, OUT_ALL=0)

ac_results, ac_success = runpf(ppc, ppopt)
dc_results, dc_success = rundcpf(ppc, ppopt)

print(f"AC PF success: {bool(ac_success)}")
print(f"DC PF success: {bool(dc_success)}")

# =========================
# 6) Build the requested tables
# =========================
def bus_table(results):
    bus_r = results["bus"]
    gen_r = results["gen"]

    # Pgenerated per bus (sum of PG from all gens at bus)
    pg_by_bus = {int(b): 0.0 for b in all_buses}
    for row in gen_r:
        b = int(row[GEN_BUS])
        pg_by_bus[b] += float(row[PG])

    rows = []
    for row in bus_r:
        b = int(row[BUS_I])
        rows.append({
            "Bus": b,
            "Pgenerated (MW)": pg_by_bus[b],
            "Pload (MW)": float(row[PD]),
        })
    return pd.DataFrame(rows).sort_values("Bus").reset_index(drop=True)

def line_table(results):
    br = results["branch"]
    rows = []
    # In PF results, PF is real power at "from" end, PT at "to" end (MW)
    for row in br:
        rows.append({
            "from (bus)": int(row[F_BUS]),
            "to (bus)": int(row[T_BUS]),
            "line flow PF (MW)": float(row[PF])  # from-end real power
        })
    return pd.DataFrame(rows).reset_index(drop=True)

# AC tables
ac_bus_tbl  = bus_table(ac_results)
ac_line_tbl = line_table(ac_results)

# DC tables
dc_bus_tbl  = bus_table(dc_results)
dc_line_tbl = line_table(dc_results)

print("\n=== Table 1 (AC) bus results ===")
display(ac_bus_tbl)

print("\n=== Table 2 (AC) line results ===")
display(ac_line_tbl)

print("\n=== Table 1 (DC) bus results ===")
display(dc_bus_tbl)

print("\n=== Table 2 (DC) line results ===")
display(dc_line_tbl)

# =========================
# 7) (Optional) quick comparison columns
# =========================
cmp_bus = ac_bus_tbl.merge(dc_bus_tbl, on="Bus", suffixes=(" (AC)", " (DC)"))
cmp_bus["ΔPgenerated (MW)"] = cmp_bus["Pgenerated (MW) (AC)"] - cmp_bus["Pgenerated (MW) (DC)"]
print("\n=== Optional: AC vs DC bus Pgenerated difference ===")
display(cmp_bus[["Bus", "Pgenerated (MW) (AC)", "Pgenerated (MW) (DC)", "ΔPgenerated (MW)"]])

AC PF success: True
DC PF success: True

=== Table 1 (AC) bus results ===


,Bus,Pgenerated (MW),Pload (MW)
0,1,108.340095,0.0
1,2,100.000000,0.0
2,3,100.000000,0.0
3,4,0.000000,100.0
4,5,0.000000,100.0
5,6,0.000000,100.0



=== Table 2 (AC) line results ===


,from (bus),to (bus),line flow PF (MW)
0,1,2,18.174815
1,1,4,50.064782
2,1,5,40.100499
3,2,3,-0.104396
4,2,4,57.148739
5,2,5,26.193645
6,2,6,34.532845
7,3,5,29.393896
8,3,6,70.501702
9,4,5,4.268301



=== Table 1 (DC) bus results ===


,Bus,Pgenerated (MW),Pload (MW)
0,1,100.0,0.0
1,2,100.0,0.0
2,3,100.0,0.0
3,4,0.0,100.0
4,5,0.0,100.0
5,6,0.0,100.0



=== Table 2 (DC) line results ===


,from (bus),to (bus),line flow PF (MW)
0,1,2,16.986075
1,1,4,46.172375
2,1,5,36.841550
3,2,3,-0.675502
4,2,4,58.372600
5,2,5,25.517500
6,2,6,33.771476
7,3,5,30.092791
8,3,6,69.231707
9,4,5,4.544975



=== Optional: AC vs DC bus Pgenerated difference ===


,Bus,Pgenerated (MW) (AC),Pgenerated (MW) (DC),ΔPgenerated (MW)
0,1,108.340095,100.0,8.340095
1,2,100.000000,100.0,0.000000
2,3,100.000000,100.0,0.000000
3,4,0.000000,0.0,0.000000
4,5,0.000000,0.0,0.000000
5,6,0.000000,0.0,0.000000
